# Working with parquet files

## Objective

+ In this assignment, we will use the data downloaded with the module `data_manager` to create features.

(11 pts total)

## Prerequisites

+ This notebook assumes that price data is available to you in the environment variable `PRICE_DATA`. If you have not done so, then execute the notebook `01_materials/labs/2_data_engineering.ipynb` to create this data set.


+ Load the environment variables using dotenv. (1 pt)

In [12]:
%load_ext dotenv
%dotenv 

The dotenv extension is already loaded. To reload it, use:
  %reload_ext dotenv


In [13]:
!pip install python-dotenv
!pip install dask

In [22]:
import dask.dataframe as dd

+ Load the environment variable `PRICE_DATA`.
+ Use [glob](https://docs.python.org/3/library/glob.html) to find the path of all parquet files in the directory `PRICE_DATA`.

(1pt)

In [21]:
import os
from glob import glob

print(os.getenv("PRICE_DATA"))

#get the PRICE_DATA directory from env variable
PRICE_DATA = os.getenv("PRICE_DATA")

#find all parquet files inside the directory (../../05_src/data/prices/)
parquet_files = glob(os.path.join(PRICE_DATA, "**/*.parquet"), recursive = True) #by making recursive=True, we can go different layers of file systems

print(parquet_files)

../../05_src/data/prices/
['../../05_src/data/prices\\ACN\\ACN_2001\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2001\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2002\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2003\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2004\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2005\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2006\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2007\\part.1.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.0.parquet', '../../05_src/data/prices\\ACN\\ACN_2008\\part.1.parquet', '../../05_src/data/prices\\AC

For each ticker and using Dask, do the following:

+ Add lags for variables Close and Adj_Close.
+ Add returns based on Close:
    
    - `returns`: (Close / Close_lag_1) - 1

+ Add the following range: 

    - `hi_lo_range`: this is the day's High minus Low.

+ Assign the result to `dd_feat`.

(4 pt)

In [77]:
#1) Read all parquet files with Dask ; Result is a Dask DataFrame with ticker column as the index
dd_px = dd.read_parquet(parquet_files, engine="pyarrow").set_index("ticker")

#check this Dask dataframe 
dd_px.info()

#check which datatype 'Date' column is
dd_px["Date"].dtype #result is: <M8[ns] ; which is datetime64[ns]

dd_px.head()

<class 'dask.dataframe.core.DataFrame'>
Columns: 9 entries, Date to Year
dtypes: datetime64[ns](1), float64(6), int32(1), string(1)

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year
ticker,,,,,,,,,
ACN,2017-01-03,117.379997,117.809998,115.820000,116.459999,110.311638,2351600.0,ACN.csv,2017
ACN,2017-01-04,116.910004,117.800003,116.430000,116.739998,110.576851,2639800.0,ACN.csv,2017
ACN,2017-01-05,116.980003,117.139999,114.949997,114.989998,108.919258,3685400.0,ACN.csv,2017
ACN,2017-01-06,114.989998,116.739998,114.339996,116.300003,110.160088,4125300.0,ACN.csv,2017
ACN,2017-01-09,116.139999,116.349998,114.919998,115.000000,108.928719,2550800.0,ACN.csv,2017


In [78]:
dd_shift = dd_px.groupby('ticker', group_keys=False).apply(
    lambda x: x.assign(Close_lag_1 = x['Close'].shift(1),
    Adj_Close_lag_1 = x['Adj Close'].shift(1))
)

C:\Users\kyuhw\AppData\Local\Temp\ipykernel_14720\1574691086.py:1: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .apply(func)
  After:  .apply(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .apply(func, meta=('x', 'f8'))            for series result
  dd_shift = dd_px.groupby('ticker', group_keys=False).apply(


In [79]:
dd_rets = dd_shift.assign(
    Returns = lambda x: x['Close']/x['Close_lag_1'] - 1,
    Adj_Returns = lambda y: y['Adj Close']/y['Adj_Close_lag_1'] - 1,
    hi_lo_range = lambda z: z['High'] - z['Low']
)

+ Convert the Dask data frame to a pandas data frame. 
+ Add a new feature containing the moving average of `returns` using a window of 10 days. There are several ways to solve this task, a simple one uses `.rolling(10).mean()`.

(3 pt)

In [107]:
"""
Dask is a lazy execution framework: commands will not execute until they are required. 
To trigger an execution in dask use `.compute()` -> this also saves to Pandas dataframe
"""
dd_rets.compute()


,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,Adj_Returns,hi_lo_range
ticker,,,,,,,,,,,,,,
ACN,2017-01-03,117.379997,117.809998,115.820000,116.459999,110.311638,2351600.0,ACN.csv,2017,NaN,NaN,NaN,NaN,1.989998
ACN,2017-01-04,116.910004,117.800003,116.430000,116.739998,110.576851,2639800.0,ACN.csv,2017,116.459999,110.311638,0.002404,0.002404,1.370003
ACN,2017-01-05,116.980003,117.139999,114.949997,114.989998,108.919258,3685400.0,ACN.csv,2017,116.739998,110.576851,-0.014991,-0.014990,2.190002
ACN,2017-01-06,114.989998,116.739998,114.339996,116.300003,110.160088,4125300.0,ACN.csv,2017,114.989998,108.919258,0.011392,0.011392,2.400002
ACN,2017-01-09,116.139999,116.349998,114.919998,115.000000,108.928719,2550800.0,ACN.csv,2017,116.300003,110.160088,-0.011178,-0.011178,1.430000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
ZIXI,2020-03-26,4.060000,4.530000,3.880000,4.510000,4.510000,1668500.0,ZIXI.csv,2020,4.000000,4.000000,0.127500,0.127500,0.650000
ZIXI,2020-03-27,4.490000,4.710000,4.100000,4.600000,4.600000,1146800.0,ZIXI.csv,2020,4.510000,4.510000,0.019956,0.019956,0.610000
ZIXI,2020-03-30,4.830000,4.870000,4.440000,4.640000,4.640000,1212000.0,ZIXI.csv,2020,4.600000,4.600000,0.008696,0.008696,0.430000


In [108]:
dd_rets["returns_ma10"] = (
    dd_rets.groupby("ticker")["Returns"]
           .transform(lambda x: x.rolling(10).mean())
)

C:\Users\kyuhw\AppData\Local\Temp\ipykernel_14720\3888428571.py:2: UserWarning: `meta` is not specified, inferred from partial data. Please provide `meta` if the result is unexpected.
  Before: .transform(func)
  After:  .transform(func, meta={'x': 'f8', 'y': 'f8'}) for dataframe result
  or:     .transform(func, meta=('x', 'f8'))            for series result
  dd_rets.groupby("ticker")["Returns"]


In [109]:
dd_rets.head(30)

,Date,Open,High,Low,Close,Adj Close,Volume,source,Year,Close_lag_1,Adj_Close_lag_1,Returns,Adj_Returns,hi_lo_range,returns_ma10
ticker,,,,,,,,,,,,,,,
ACN,2017-01-03,117.379997,117.809998,115.820000,116.459999,110.311638,2351600.0,ACN.csv,2017,NaN,NaN,NaN,NaN,1.989998,NaN
ACN,2017-01-04,116.910004,117.800003,116.430000,116.739998,110.576851,2639800.0,ACN.csv,2017,116.459999,110.311638,0.002404,0.002404,1.370003,NaN
ACN,2017-01-05,116.980003,117.139999,114.949997,114.989998,108.919258,3685400.0,ACN.csv,2017,116.739998,110.576851,-0.014991,-0.014990,2.190002,NaN
ACN,2017-01-06,114.989998,116.739998,114.339996,116.300003,110.160088,4125300.0,ACN.csv,2017,114.989998,108.919258,0.011392,0.011392,2.400002,NaN
ACN,2017-01-09,116.139999,116.349998,114.919998,115.000000,108.928719,2550800.0,ACN.csv,2017,116.300003,110.160088,-0.011178,-0.011178,1.430000,NaN
ACN,2017-01-10,114.940002,115.989998,114.639999,115.059998,108.985558,2550100.0,ACN.csv,2017,115.000000,108.928719,0.000522,0.000522,1.349998,NaN
ACN,2017-01-11,115.400002,116.260002,114.910004,116.120003,109.989594,2845800.0,ACN.csv,2017,115.059998,108.985558,0.009213,0.009213,1.349998,NaN
ACN,2017-01-12,115.620003,115.970001,114.839996,115.779999,109.667542,2110800.0,ACN.csv,2017,116.120003,109.989594,-0.002928,-0.002928,1.130005,NaN
ACN,2017-01-13,115.349998,117.440002,115.349998,116.949997,110.775764,2665600.0,ACN.csv,2017,115.779999,109.667542,0.010105,0.010105,2.090004,NaN


Please comment:

+ Was it necessary to convert to pandas to calculate the moving average return?
+ Would it have been better to do it in Dask? Why?

(1 pt)

For that question, we could have done in both Dask or Pandas Dataframe formats; not necessary. It would have been better with Dask since it is a package that enables parallel computing. And, since we originally had it in Dask dataframe format, it would have been better to have done all the necessary calculations within Dask df prior to converting to Pandas.

## Criteria

The [rubric](./assignment_1_rubric_clean.xlsx) contains the criteria for grading.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-1`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at `#cohort-3-help`. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.